## Using USB3 Camera Shield UC-593 Rev C

This is an exploration of the Arducam SDK, brought in from the original Python Demo. It works in an environment of Python up to 3.12 (with `arducam_config_parser` and `ArducamSDK` installed via `pip`), or with custom wrapper around the C/C++ dynamic libraries.

**NOTE:** The replacement for the original SDK is intended as a 1:1 replacement. If the replacement ArducamSDK is present in the "current directory", it takes precedence over the one installed in the virtual environment.

In [ ]:
import os, sys, time
from pathlib import Path
print(sys.version_info)

import arducam_config_parser
import ArducamSDK

In [ ]:
# Checks for connected cameras, returning their number, a list of indices, and a list of their serials
# There is no Python interface to interpret the serials; according ro C API, this is a list of 16 uint8
# The serial is a memoryview object, can access each component by index, 0..15. Going above raises "Index Error".
#
# Python demo code reads 16 bytes from *initialized* camera (from UserData, not from scan()), but uses only first 12. Last 4 bytes are 0.
#    rtn_val,datas = ArducamSDK.Py_ArduCam_readUserData(handle,0x400-16, 16)
scn = ArducamSDK.Py_ArduCam_scan()

print(scn)
print(scn[0], type(scn[0]))
print(scn[1], type(scn[1]), '  ----  ', scn[1][0], type(scn[1][0]))
print(scn[2], type(scn[2]), '  ----  ', scn[2][0], type(scn[2][0]), len(scn[2][0]))

# Serial processed the same way as in Demo from UserData, but all 16 bytes
print("\nSerial: %c%c%c%c-%c%c%c%c-%c%c%c%c-%c%c%c%c"%(scn[2][0][0],scn[2][0][1],scn[2][0][2],scn[2][0][3],
                                            scn[2][0][4],scn[2][0][5],scn[2][0][6],scn[2][0][7],
                                            scn[2][0][8],scn[2][0][9],scn[2][0][10],scn[2][0][11],
                                            scn[2][0][12],scn[2][0][13],scn[2][0][14],scn[2][0][15]))

# bb = scn[2][0].tobytes()    # Original SDK has memoryview
bb = scn[2][0]    # Modified SDK has bytes object
print('<memory>.tobytes():               ', bb)
# bb = scn[2][0][0:12].tobytes()  # Original demo only uses first 12 bytes/chars
print('<memory>.tobytes().decode(), processed as string:')
print(bb.decode(),  len(bb), len(bb.decode()))
# Sequencing a memoryview only creates a new memoryview object over the same memory;
# Sequencing a bytes creates a copy of the range. More convenient, but slower. Slowdown not significant for small sizes
# print('-'.join([scn[2][0][i:i+4].tobytes().decode() for i in range(0,10,4)]))
print('-'.join([bb[i:i+4].decode() for i in range(0,10,4)]))


In [ ]:
# Pick a configuration file
# %gui tk
# import tkinter
from tkinter.filedialog import askopenfilename
configName = askopenfilename() # show an "Open" dialog box and return the path to the selected file
# %gui
print("Config file:", configName, type(configName), bool(configName))

In [ ]:
configName = Path('../Config') / 'MT9J001_MONO_12b_640x480.cfg'


In [ ]:
## Prepare configuration from file to be loaded in camera
print("Loading config file ", configName)
config = arducam_config_parser.LoadConfigFile(str(configName))
camera_parameter = config.camera_param.getdict()

# Need both bitWidth and byteLength passed to the camera, even though byteLength ca
#    bitWidth tells the sensor how manu steps to use for sampling data, usually 8/10/12 bits
#    byteLength tells how the data is packed for transmission
# The following fails for bitWidth>16, but there are no such cameras; comparison with 16 is superfluous
bitWidth = camera_parameter["BIT_WIDTH"]
# Original line does not initialize for bitWidth>16, but there are no cameras with bit width above 16.
# Comparison with 16 is redundant.
byteLength = 2 if bitWidth > 8 and bitWidth <= 16 else 1
cfg = { "u32CameraType":0x00,
        "usbType":0,
        "u16Vid":0,
        "u32Size":0,
        "u32Width":camera_parameter["WIDTH"],
        "u32Height":camera_parameter["HEIGHT"],
        "u8PixelBytes":byteLength,
        "u8PixelBits":bitWidth,
        "u32I2cAddr":camera_parameter["I2C_ADDR"],
        "emI2cMode":camera_parameter["I2C_MODE"],
        "emImageFmtMode":camera_parameter["FORMAT"][0],
        "u32TransLvl":camera_parameter["TRANS_LVL"] }


In [ ]:
# General Camera configuration, complete from file, not only those passed to camera on initialization
camera_parameter = config.camera_param.getdict()
print(camera_parameter)

In [ ]:
## Can open and initialize in two ways. 
# autoopen(cfg) is identical to open(cfg,0), identical if there is only one camera
#ret,handle,rtn_cfg = ArducamSDK.Py_ArduCam_open(cfg,0)    # open specified camera, if several
ret, handle, rtn_cfg = ArducamSDK.Py_ArduCam_autoopen(cfg)    # opens first detected camera
#    ret: return error, should be 0 to proceed
#    handle: camera handle, an opaque Python Capsule
#    rtn_cfg: actual configuration of camera (updated); missing fields are initialized, others should be identical to requested cfg
print(handle, type(handle))

for key in (set(rtn_cfg) | set(cfg)):
    print(f"{key:15}  ({cfg[key]:4},   {type(cfg[key])};  {rtn_cfg[key]:4}, {type(rtn_cfg[key])})" )

In [ ]:
## Load board configuration and sensor registers to camera
command_names = {arducam_config_parser.CONFIG_TYPE_REG: 'REG',
                 arducam_config_parser.CONFIG_TYPE_DELAY: 'DELAY',
                 arducam_config_parser.CONFIG_TYPE_VRCMD: 'VRCMD'}
usb_version = rtn_cfg['usbType']
for i in range(config.configs_length):
    # configs[i].type is an int, 4 bytes
    #    Top 8 (type >> 24): section type
    #    Mid 8 ((type >> 16) & 0xFF): USB type (2, 3_3, 3_2; 0 if unspecified)
    #    Low 16 (type & 0xFFFF): command (VRCMD, DELAY, REG)
    # The most significant byte is set to 2 for VRCMD and 3 for the others, does not seem to matter
    # The second MSB is the USB version, 
    cfgType = config.configs[i].type
    
    skip = False    # skip if USB type specified and deos not match what camera reported
    if ((cfgType >> 16) & 0xFF) != 0 and ((cfgType >> 16) & 0xFF) != usb_version: skip=True

    if not skip:
        if cfgType & 0xFFFF == arducam_config_parser.CONFIG_TYPE_REG:
            ArducamSDK.Py_ArduCam_writeSensorReg(handle, config.configs[i].params[0], config.configs[i].params[1])
        elif cfgType & 0xFFFF == arducam_config_parser.CONFIG_TYPE_DELAY:
            time.sleep(float(config.configs[i].params[0])/1000)
        elif cfgType & 0xFFFF == arducam_config_parser.CONFIG_TYPE_VRCMD:
            ArducamSDK.Py_ArduCam_setboardConfig(handle, config.configs[i].params[0],
                                                 config.configs[i].params[1], config.configs[i].params[2], config.configs[i].params[3],
                                                 config.configs[i].params[4: config.configs[i].params_length])

In [ ]:
## Check serial via UserData
# `datas` is a memoryview, same as ther serial number returned by `scan()`
rtn_val,datas = ArducamSDK.Py_ArduCam_readUserData(handle,0x400-16, 16)
print("Serial: %c%c%c%c-%c%c%c%c-%c%c%c%c"%(datas[0],datas[1],datas[2],datas[3],
                                            datas[4],datas[5],datas[6],datas[7],
                                            datas[8],datas[9],datas[10],datas[11]))
# ddata = datas.tobytes().decode()    # Serial in datas is memoryview in original SDK
ddata = datas.decode()              # Serial in datas is bytes here
print(ddata, ' --> ',  '-'.join([ddata[i:i+4] for i in range(0,10,4)]))

In [ ]:
## readReg tests from the C++ demo used to retrieve firmware info -- from sensor?
USB_CPLD_I2C_ADDRESS = 0x46
ret, version = ArducamSDK.Py_ArduCam_readReg_8_8(handle, USB_CPLD_I2C_ADDRESS, 0x00)
print(f"{ret=}, {version=}, {hex(version)=},   v{version>>4}.{version & 0x0F}")
ret, year    = ArducamSDK.Py_ArduCam_readReg_8_8(handle, USB_CPLD_I2C_ADDRESS, 0x05)
print(f"{ret=}, {year=}")
ret, month   = ArducamSDK.Py_ArduCam_readReg_8_8(handle, USB_CPLD_I2C_ADDRESS, 0x06)
print(f"{ret=}, {month=}")
ret, day     = ArducamSDK.Py_ArduCam_readReg_8_8(handle, USB_CPLD_I2C_ADDRESS, 0x07)
print(f"{ret=}, {day=}")

In [ ]:
## Firmware version, retrieved from the board
ret, data = ArducamSDK.Py_ArduCam_getboardConfig(handle, 0x80, 0x00, 0x00, 2)
print(ret, data)
print(f"Firmware version: v{data[0]}.{data[1]}")

## Outline of operation

Normal (streaming) use:

1. Start a ***capture image thread***:
   * Start capture: `beginCaptureImage()`
   * Capture: `captureImage()`. Each successful call results in an image stored in the buffer, up to maximum capacity. The call returns an error code.
   * End capture when done: `endCaptureImage()`
1. Start a ***read image thread***:
   * Check if any image is available; `availableImage()` returns the number of available in the buffer
   * Read the image from buffer, `readImage()` and process it
   * Delete the image from the buffer, `del()`, to make space for a new capture
   * When completely done, call `flush()` to completely clean the camera buffer

It is also possible to run capture without threading; start capture as usual, and run in loop `captureImage()`, followed by check and read the image.

**NOTES:** 

1. An image may still be captured for some error codes (e.g., `USB_CAMERA_FRAME_INDEX_ERROR`, code `0xFF25`), although the image may be corrupt.
1. It is normal to have some errors at the beginning (e.g., `USB_CAMERA_DATA_LEN_ERROR`, code `0xFF24`). This usually vanishes as parameters take effect.

In [ ]:
ArducamSDK.Py_ArduCam_setMode(handle,ArducamSDK.CONTINUOUS_MODE)
rtn_val = ArducamSDK.Py_ArduCam_beginCaptureImage(handle)
print("Begin Capture: ",hex(rtn_val))
for i in range(40):
    rtn_val = ArducamSDK.Py_ArduCam_captureImage(handle)
    if rtn_val<255:
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            err_code, data, img_cfg = ArducamSDK.Py_ArduCam_readImage(handle)
            print(f"       Image Captured: {err_code:#06X}")
            ArducamSDK.Py_ArduCam_del(handle)
            break
    else:
        # an image may be retrieved even if an error code is returnedm but may be incomplete
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            ArducamSDK.Py_ArduCam_del(handle)
        
    print(f"{i:3}    Capture Image: {rtn_val:#06X}    Available images: {ArducamSDK.Py_ArduCam_availableImage(handle)}")

ArducamSDK.Py_ArduCam_flush(handle)
rtn_val = ArducamSDK.Py_ArduCam_endCaptureImage(handle)
print("End Capture: ",hex(rtn_val))


If the raw data has more than 8bpp, it needs to be aligned with the usual data types for display: shift right or left so its most significant bit is aligned with the most significant bit of the type.

For processing of the retrieved raw data, we need to use OpenCV. Note that OpenCV `cv2.imshow()` does not seem to work in Jupyter or from interactive console, although it works just fine when used in a program run from console.

Alternatives:

1. `IPython.display.Image()` -- requires a file name or the data as if read from file, in an appropriate format (e.g., PNG), . Can also specify size as argument, to scale the image
2. `matplotlib.pyplot.imshow()` -- most flexible, can have automatic scaling and color map
3. `tkinter` -- image needs to be in appropriate format (`PhotoImage`), added to a supporting widget (`label` is simplest, `canavs` is the most flexible -- can arbitrarily place, zoom/scroll). `tkinter` does not display the image inline, but pops a separate window.
    1. Loaded directly from a restricted number of files (`gif` and `png` are supported), `PhotoImage(file=...`)
    2. Pass data content as if read from file, but must be encoded as base64 (`PhotoImage(data=base64.b64encode(buffer))`)
    3. Generate `ImageTk` by PIL (more flexible), which can be used directly where `tkinter` can use an image. 

In [ ]:
import numpy as np
import cv2
from IPython.display import Image
import matplotlib.pyplot as plt

In [ ]:
# Need to reshape the image stream into array with correct size for image

# 12bpp, shifted down to 1 byte per pixel, 12bpp->8bpp
# image = (np.frombuffer(data, np.uint16).reshape(img_cfg['u32Height'], img_cfg['u32Width']) >> (img_cfg['u8PixelBits']-8))
# image = image.astype(np.uint8)

# 12bpp, 2 byte per pixel, shifted up 12bpp->16bpp
# image = (np.frombuffer(data, np.uint16).reshape(img_cfg['u32Height'], img_cfg['u32Width']) << (16 - img_cfg['u8PixelBits']))

# 8bpp
image = np.frombuffer(data, np.uint8).reshape(img_cfg['u32Height'], img_cfg['u32Width'])

# image = cv2.cvtColor(image, cv2.COLOR_BayerRG2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGR2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGB2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerBG2BGR)

# Encode the array to PNG and show it directly with IPython
retval, buffer = cv2.imencode('.png', image)
Image(data=buffer)
# Image(data=buffer, width=1500)

In [ ]:
%gui tk
import tkinter
import base64

root = tkinter.Tk()
root.title("Arducam Image")
root.geometry(f"{img_cfg['u32Width']}x{img_cfg['u32Height']}")
image_tk = tkinter.PhotoImage(data=base64.b64encode(buffer))
image_label = tkinter.Label(root, image=image_tk)
image_label.pack()

root.mainloop()
%gui

In [ ]:
# plt.figure(figsize=(15, 15))
plt.imshow(image, aspect=1, cmap='gray')
plt.axis('off')

In [ ]:
plt.imshow(image[:50,:100], aspect=1)
plt.axis('off')

In [ ]:
ArducamSDK.Py_ArduCam_close(handle)